In [1]:
import sqlite3
import pandas as pd

DB_NAME = "cat_fleet_medallion.db"
conn = sqlite3.connect(DB_NAME)

# load gold table from yesterday
df = pd.read_sql("SELECT * FROM gold_telematics", conn)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['machine_id', 'timestamp']).reset_index(drop=True)

# ── 1. LAG FEATURES ────────────────────────────────────────────────────────
lag_cols = [
    'engine_temp_c', 'engine_rpm', 'oil_pressure_psi', 'coolant_temp_c',
    'vibration_level', 'battery_voltage_v', 'brake_temp_c', 'brake_pad_wear_mm'
]
for col in lag_cols:
    df[f'{col}_lag1'] = df.groupby('machine_id')[col].shift(1)

# ── 2. DELTA FEATURES ──────────────────────────────────────────────────────
for col in lag_cols:
    df[f'{col}_delta1'] = df[col] - df[f'{col}_lag1']

# ── 3. CUMULATIVE IDLING HOURS (using REAL time gaps, not assumed) ───────
df['time_diff_hours'] = df.groupby('machine_id')['timestamp'].diff().dt.total_seconds() / 3600
df['idling_hours_this_reading'] = df['is_idling'] * df['time_diff_hours'].fillna(0)
df['cumulative_idling_hours'] = df.groupby('machine_id')['idling_hours_this_reading'].cumsum()

# ── 4. ADD MISSING ROLLING AVERAGES (brake + other sensors) ──────────────
extra_roll_cols = ['engine_temp_c', 'coolant_temp_c', 'vibration_level',
                    'brake_temp_c', 'brake_pad_wear_mm', 'brake_fluid_level_psi']
for col in extra_roll_cols:
    df[f'{col}_roll3_mean'] = (
        df.groupby('machine_id')[col]
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )

# ── 5. RE-SAVE ENRICHED GOLD TABLE ────────────────────────────────────────
df.to_sql("gold_telematics", conn, if_exists="replace", index=False)
print(f"Gold table enriched and saved: {len(df)} rows, {len(df.columns)} columns")

# ── AUDIT ──────────────────────────────────────────────────────────────────
sample = pd.read_sql(
    "SELECT machine_id, timestamp, engine_temp_c, engine_temp_c_lag1, "
    "engine_temp_c_delta1, time_diff_hours, cumulative_idling_hours FROM gold_telematics LIMIT 8",
    conn
)
print(sample.to_string(index=False))

conn.close()

Gold table enriched and saved: 1970 rows, 75 columns
machine_id           timestamp  engine_temp_c  engine_temp_c_lag1  engine_temp_c_delta1  time_diff_hours  cumulative_idling_hours
   VEH0000 2023-01-01 00:00:00      99.388213                 NaN                   NaN              NaN                      0.0
   VEH0000 2023-01-01 00:56:00      84.118706           99.388213            -15.269507         0.933333                      0.0
   VEH0000 2023-01-01 01:00:00      96.887263           84.118706             12.768558         0.066667                      0.0
   VEH0000 2023-01-01 01:12:00     105.465312           96.887263              8.578049         0.200000                      0.0
   VEH0000 2023-01-01 01:27:00      91.130527          105.465312            -14.334785         0.250000                      0.0
   VEH0000 2023-01-01 01:28:00     103.521431           91.130527             12.390904         0.016667                      0.0
   VEH0000 2023-01-01 02:06:00      9